## Creating a gold dimension view

In [0]:
%sql
CREATE OR REPLACE VIEW transport_lakehouse.gold.dim_vehicle_type AS
SELECT
  ROW_NUMBER() OVER (ORDER BY vehicle_type_cd, vehicle_type_nm) AS vehicle_type_key,
  vehicle_type_cd,
  vehicle_type_nm,
  MAX(vehicle_type_eu_cd) AS vehicle_type_eu_cd
FROM (
  SELECT
    CAST(vehicle_type_cd AS STRING) AS vehicle_type_cd,
    vehicle_type_nm,
    CAST(vehicle_type_eu_cd AS STRING) AS vehicle_type_eu_cd
  FROM transport_lakehouse.silver.silver_vehicles_public

  UNION ALL

  SELECT
    CAST(vehicle_type_cd AS STRING) AS vehicle_type_cd,
    vehicle_type_nm,
    NULL AS vehicle_type_eu_cd
  FROM transport_lakehouse.silver.silver_vehicles_two_wheeled
) t
WHERE vehicle_type_cd IS NOT NULL AND vehicle_type_nm IS NOT NULL
GROUP BY vehicle_type_cd, vehicle_type_nm


## Quality check

In [0]:
%sql
select * 
from transport_lakehouse.gold.dim_vehicle_type

In [0]:
%sql
SELECT
  vehicle_type_cd,
  vehicle_type_nm,
  COUNT(*) cnt
FROM transport_lakehouse.gold.dim_vehicle_type
GROUP BY vehicle_type_cd, vehicle_type_nm
HAVING COUNT(*) > 1
ORDER BY cnt DESC;
